In [1]:
# 1. Install required packages
# Run this once in your environment
# pip install kaggle pandas sqlite3

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
# %pip install seaborn

In [2]:
#print(os.getcwd())

In [3]:
# 3. Load CSV into pandas
#df_ops = pd.read_csv('teleco_customers.csv')

In [4]:
df_ops.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
df_ops.shape

(7043, 21)

#### Data Preparation and Cleaning

##### **1. Check information schema table**
```
SELECT column_name, data_type 
FROM information_schema.columns 
WHERE table_name = 'TelcoCustomers';
```

This query helps identify the type of each column, aiding in understanding which columns might need transformation or cleaning.

##### **2. Missing Values**

* Missing values can skew analysis results or hinder model training.
* In SQL, missing values can be handled by setting default values.

```
UPDATE TelcoCustomers 
SET TotalCharges = 0 
WHERE TotalCharges IS NULL;
```

##### **3. Identify and Delete Duplicate records**

```
SELECT customerID, COUNT(*) 
FROM TelcoCustomers 
GROUP BY customerID 
HAVING COUNT(*) > 1;
```

```
DELETE FROM TelcoCustomers 
WHERE ROWID NOT IN 
    (SELECT MIN(ROWID) 
    FROM TelcoCustomers 
    GROUP BY customerID);
```

##### **4. Transform variables**

* This involves normalizing or standardizing numerical data and converting categorical data to a format suitable for analysis.
```
UPDATE TelcoCustomers 
SET SeniorCitizen = CASE 
    WHEN SeniorCitizen = 1 THEN 'Yes' 
    ELSE 'No' 
END;
```

##### **5. Outlier Detection**

* Outliers can substantially distort statistical analyses and machine learning models.
* Identifying outliers involves statistical methods like calculating Z-scores or IQR ranges to flag unusual data points:

```
SELECT customerID, MonthlyCharge 
FROM TelcoCustomers 
WHERE MonthlyCharge > (SELECT AVG(MonthlyCharge) + 3 * STDDEV(MonthlyCharge) FROM TelcoCustomers);
```

* Cleaning outliers may involve capping, flooring, or entirely removing them from the analysis set.



In [6]:
conn = sqlite3.connect('TelcoCustomers.db')
df_ops.to_sql('TelcoCustomers', conn, index=False, if_exists='replace')

7043

In [7]:
def query_and_print(sql):
    result = pd.read_sql(sql, conn)
    print(result, '\n')

In [8]:
analysis_queries_data_prep = {
    'Task 1: Count Rows': '''
        SELECT 
        *
        FROM TelcoCustomers LIMIT 1
    ''',
    
    'Task 2: EDA - Identify and Remove duplicate records if any': ''' 
    SELECT customerID, COUNT(*) 
    FROM TelcoCustomers 
    GROUP BY customerID 
    HAVING COUNT(*) > 1;
    ''' ,

    'Task 5: Outlier Detection': '''
    SELECT customerID, MonthlyCharges 
    FROM TelcoCustomers 
    WHERE MonthlyCharges > (SELECT AVG(MonthlyCharges) + 3 * SQRT(AVG(MonthlyCharges * MonthlyCharges) - AVG(MonthlyCharges) * AVG(MonthlyCharges) ) FROM TelcoCustomers);
    '''
}

In [9]:
for desc, sql in analysis_queries_data_prep.items():
    print(f'-- {desc} --')
    query_and_print(sql)

-- Task 1: Count Rows --
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling  \
0          No          No              No  Month-to-month              Yes   

      PaymentMethod MonthlyCharges  TotalCharges Churn  
0  Electronic check          29.85         29.85    No  

[1 rows x 21 columns] 

-- Task 2: EDA - Identify and Remove duplicate records if any --
Empty DataFrame
Columns: [customerID, COUNT(*)]
Index: [] 

-- Task 5: Outlier Detection --
Empty DataFrame
Columns: [customerID, MonthlyCharges]
Index: [] 



In [14]:
analysis_queries_eda_contd = {
    'Task 1: Data Distributions': '''
        SELECT 
    AVG(tenure) AS avg_tenure,
    MIN(tenure) AS min_tenure,
    MAX(tenure) AS max_tenure,
    AVG(MonthlyCharges) AS avg_monthly_charges,
    MIN(MonthlyCharges) AS min_monthly_charges,
    MAX(MonthlyCharges) AS max_monthly_charges
    FROM TelcoCustomers;
    ''',
    
    'Task 2: EDA - Categorical Distribution': ''' 
    SELECT Contract, COUNT(*) AS count
    FROM TelcoCustomers
    GROUP BY Contract;
    ''' ,

    'Task 3: Correlations': '''
    SELECT 
    tenure, 
    AVG(MonthlyCharges) AS avg_monthly_charges
    FROM TelcoCustomers
    GROUP BY tenure
    ORDER BY tenure;
    '''
}

for desc, sql in analysis_queries_eda_contd.items():
    print(f'-- {desc} --')
    query_and_print(sql)

-- Task 1: Data Distributions --
   avg_tenure  min_tenure  max_tenure  avg_monthly_charges  \
0   32.371149           0          72            64.761692   

   min_monthly_charges  max_monthly_charges  
0                18.25               118.75   

-- Task 2: EDA - Categorical Distribution --
         Contract  count
0  Month-to-month   3875
1        One year   1473
2        Two year   1695 

-- Task 3: Correlations --
    tenure  avg_monthly_charges
0        0            41.418182
1        1            50.485808
2        2            57.206303
3        3            58.015000
4        4            57.432670
..     ...                  ...
68      68            73.321000
69      69            70.823158
70      70            76.378992
71      71            73.735588
72      72            80.695856

[73 rows x 2 columns] 



#### Exploratory Data Analysis (EDA) with SQL

##### **1. Data Distribution**
Lets understand the tenure and MonthlyCharges, to understand customer behavior. 
```
SELECT 
    AVG(tenure) AS avg_tenure,
    MIN(tenure) AS min_tenure,
    MAX(tenure) AS max_tenure,
    AVG(MonthlyCharges) AS avg_monthly_charges,
    MIN(MonthlyCharges) AS min_monthly_charges,
    MAX(MonthlyCharges) AS max_monthly_charges
FROM TelcoCustomers;
```

##### **2. Categorical distributions**

Understanding the distribution of categorical features, such as Contract, PaymentMethod, or InternetService, can reveal the prevalence of various customer preferences.

```
SELECT Contract, COUNT(*) AS count
FROM TelcoCustomers
GROUP BY Contract;
```

##### **3. Correlations** 
MonthlyCharges vs  tenure, can unveil potential patterns driving customer churn.

```
SELECT 
    tenure, 
    AVG(MonthlyCharges) AS avg_monthly_charges
FROM TelcoCustomers
GROUP BY tenure
ORDER BY tenure;
```

In [12]:
analysis_queries_eda = {
    'Task 1: Data Distribution': '''
        SELECT 
        AVG(tenure) AS avg_tenure,
        MIN(tenure) AS min_tenure,
        MAX(tenure) AS max_tenure,
        AVG(MonthlyCharges) AS avg_monthly_charges,
        MIN(MonthlyCharges) AS min_monthly_charges,
        MAX(MonthlyCharges) AS max_monthly_charges
        FROM TelcoCustomers;
    ''',
    
    'Task 2: Categorical Distribution': ''' 
    SELECT customerID, COUNT(*) 
    FROM TelcoCustomers 
    GROUP BY customerID 
    HAVING COUNT(*) > 1;
    ''' ,

    'Task 3: Correlations': '''
    SELECT 
    tenure, 
    AVG(MonthlyCharges) AS avg_monthly_charges
    FROM TelcoCustomers
    GROUP BY tenure
    ORDER BY tenure;
    '''
}

In [13]:
for desc, sql in analysis_queries_eda.items():
    print(f'-- {desc} --')
    query_and_print(sql)

-- Task 1: Data Distribution --
   avg_tenure  min_tenure  max_tenure  avg_monthly_charges  \
0   32.371149           0          72            64.761692   

   min_monthly_charges  max_monthly_charges  
0                18.25               118.75   

-- Task 2: Categorical Distribution --
Empty DataFrame
Columns: [customerID, COUNT(*)]
Index: [] 

-- Task 3: Correlations --
    tenure  avg_monthly_charges
0        0            41.418182
1        1            50.485808
2        2            57.206303
3        3            58.015000
4        4            57.432670
..     ...                  ...
68      68            73.321000
69      69            70.823158
70      70            76.378992
71      71            73.735588
72      72            80.695856

[73 rows x 2 columns] 



#### EDA Contd... 

##### **1. Detecting Churn Patterns**
- EDA aims to uncover any relationships between customer characteristics and churn. 

```
SELECT 
    InternetService, 
    AVG(CAST(Churn AS INT)) AS churn_rate
FROM TelcoCustomers
GROUP BY InternetService;
```

##### **1. Summary Statistics for Key Features** 

```
SELECT
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churn_count,
    AVG(Monthly_Charge) AS avg_monthly_charges,
    AVG(Total_Charges) AS avg_total_charges
FROM TelcoCustomers;
```

**Insights** 
- The statistics produced here can serve as benchmarks when investigating specific customer segments that display higher churn tendencies.
- Insights drawn assist in forming more refined hypotheses and feature selection during the modeling phase.
- By continuously querying and analyzing different aspects, EDA with SQL enables data-driven narratives that guide strategic decisions, reinforcing customer retention efforts.

#### Advanced SQL Analysis for Churn Analysis

**1. Complex Joins and Subqueries**

```
SELECT t.CustomerID, t.Churn, c.ComplaintID
FROM TelcoCustomers t
LEFT JOIN CustomerComplaints c ON t.CustomerID = c.CustomerID;
```

- Helps to identify customers who have lodged complaints and correlate these with churn status, highlighting a potential trigger for customer attrition.
- Subqueries also play a vital role in breaking down complex logic into manageable parts. 

```
SELECT InternetService, AVG(tenure) AS avg_tenure
FROM (
    SELECT tenure, InternetService
    FROM TelcoCustomers
    WHERE Churn = 'Yes'
) AS Churned_Customers
GROUP BY InternetService;
```

**2. Window Functions for Detailed Insights**

- Lets say you want to find tenure rank of each customer then you can use the below query.

```
SELECT CustomerID, tenure, 
RANK() OVER (ORDER BY tenure DESC) AS tenure_rank 
FROM TelcoCustomers;
```
- This query ranks customers by their length of tenure, providing insights into those most at risk of churn based on shorter engagements.

**3. CTEs (Common Table Expressions) for Simplification**

- CTEs enable writing more readable SQL queries.
- For instance, analyzing churn rates among different demographics can be more systematic with CTEs.

```
WITH Demographics AS (
    SELECT CustomerID, Gender, SeniorCitizen, Churn
    FROM TelcoCustomers
),
ChurnedSeniorCitizens AS (
    SELECT Gender, COUNT(*) AS senior_churn_count
    FROM Demographics
    WHERE SeniorCitizen = 1 AND Churn = 'Yes'
    GROUP BY Gender
)
SELECT *
FROM ChurnedSeniorCitizens;
```

**4. REGEX for Pattern matching**

- If billing query comments are stored in a BillingQueries table, we can identify customers frequently querying about late fees:
- Identifying such patterns helps to track the frequency of certain concerns and their impact on churn

```
SELECT CustomerID, COUNT(QueryID)
FROM BillingQueries
WHERE Comment LIKE '%late fee%'
GROUP BY CustomerID;
```


#### Advanced Analytical constraints

##### **1. Annual subscription plan changes**
- By combining constraints within complex queries, telcos can pinpoint precise customer segments at risk.

```
SELECT CustomerID, COUNT(*) AS service_changes
FROM ServiceChanges
WHERE ChangeType = 'Plan Downgrade'
AND Date > (SELECT DATEADD(month, -6, CURRENT_DATE))
GROUP BY CustomerID
HAVING COUNT(*) > 2;
```
- Potential prelude to churn.
- By utilizing these advanced SQL techniques, organizations can enhance their ability to interpret intricate trends, drive strategic interventions, and positively influence customer retention strategies.


#### Key Metrics for visualizations

##### **1. Churn rate for different categories**

- Churn rate for different categories
- 
```
SELECT 
    InternetService, 
    ROUND(AVG(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100, 2) AS churn_rate
FROM 
    TelcoCustomers
GROUP BY 
    InternetService;
```

- This query gives the churn rate percentage for each internet service type. Bar charts, pie charts can be used for visualization in Tableau or other BI.

##### **2. Time-bound Analysis for Trends**

- Churn trends over time (year, months)
- 
```
SELECT 
    EXTRACT(YEAR FROM DepositDate) AS year, 
    EXTRACT(MONTH FROM DepositDate) AS month, 
    COUNT(*) as total_customers, 
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) as churned_customers,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as churn_rate
FROM 
    TelcoCustomers
GROUP BY 
    year, month
ORDER BY 
    year, month;
```

##### **3. Churn trends by location, gender, age etc**

```
SELECT 
    CASE 
        WHEN SeniorCitizen = 1 THEN 'Senior' 
        ELSE 'Non-Senior' 
    END AS age_group, 
    ROUND(AVG(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100, 2) AS churn_rate
FROM 
    TelcoCustomers
GROUP BY 
    age_group;
```

##### **4. Utilizing Heatmaps for Service Impact**

- **Heatmaps for correlations between different services and churn**
```
SELECT 
    InternetService, 
    PhoneService, 
    ROUND(AVG(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100, 2) AS churn_rate
FROM 
    TelcoCustomers
GROUP BY 
    InternetService, PhoneService;
```


In [15]:
conn.close()